# Local RAG AI (MacBook)
This notebook builds a **local AI with constantly updated knowledge**.

Pipeline:

Internet → Crawl → Clean text → Embedding → FAISS Vector DB → Retrieve → Ask Local LLM

Recommended stack:
- Ollama (for local LLM)
- Mistral 7B
- SentenceTransformers
- FAISS


## 1. Install Dependencies
Run once.

In [ ]:
# pip install requests beautifulsoup4 sentence-transformers faiss-cpu ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 41.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 48.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 44.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [sentence-transformers]ence-transformers]
Note: you may need to restart the kernel to use updated packages.


## 2. Import Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import ollama


/Users/wondongsoo/miniconda3/conda_envs/AI/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Web Crawler

In [2]:
def crawl(url):
    r = requests.get(url)
    soup = BeautifulSoup(r.text, 'html.parser')

    texts = []
    for p in soup.find_all('p'):
        texts.append(p.get_text())

    return '\n'.join(texts)

## 4. Collect Documents

In [3]:
urls = [
    'https://news.ycombinator.com/',
    'https://arxiv.org/'
]

documents = []

for url in urls:
    print('Crawling:', url)
    documents.append(crawl(url))

print('Documents collected:', len(documents))

Crawling: https://news.ycombinator.com/
Crawling: https://arxiv.org/
Documents collected: 2


## 5. Embedding Model

In [4]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1469.81it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 6. Create Embeddings

In [5]:
embeddings = embed_model.encode(documents)
embeddings = np.array(embeddings)

## 7. Build Vector Database (FAISS)

In [6]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Vector DB size:', index.ntotal)

Vector DB size: 2


## 8. Search Function

In [7]:
def search(query, k=3):
    q = embed_model.encode([query])
    q = np.array(q)

    D, I = index.search(q, k)

    results = []
    for i in I[0]:
        results.append(documents[i])

    return results

## 9. Ask Local LLM (Ollama)

In [8]:
def ask_llm(question):

    context = search(question)

    prompt = f"""
Context:
{context}

Question:
{question}
"""

    response = ollama.chat(
        model='mistral',
        messages=[{'role':'user','content':prompt}]
    )

    return response['message']['content']

## 10. Test Query

In [9]:
print(ask_llm('What are the latest AI research trends?'))

 Since the provided text is about arXiv, an open-access archive, and not specific to the latest AI research trends, I cannot provide a direct answer from the text. However, arXiv covers several fields including Computer Science, which often includes AI research. To find the latest AI research trends, you may want to search for recently published AI articles on arXiv or visit other reputable AI research platforms like AI Crows or AI Trends.
